# StatPitch — Model Improvements
## Notebook 5 — Complete Upgrade

---

We implement every improvement in order of impact:

| Part | What | Expected gain |
|---|---|---|
| **A** | Fix H2H in API + rest days + goal momentum | +2–3% |
| **B** | Hyperparameter tuning with Optuna | +2–4% |
| **C** | Dixon-Coles low-score correction | +2–4% on markets |
| **D** | Probability calibration | Better log loss |
| **E** | Model stacking (meta-learner) | +1–2% |
| **F** | Neural network (MLP) | Adds diversity to ensemble |

At the end we run a full before-vs-after comparison on the 2022 World Cup.

---
## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR    = '/content/drive/MyDrive/StatPitch'
PROJECT_DIR = os.path.join(SAVE_DIR, 'statpitch_api')
os.chdir(SAVE_DIR)

!pip install optuna --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib, warnings
from datetime import datetime
from scipy.stats import poisson
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.model_selection import cross_val_predict, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'

print('All libraries loaded!')

In [ ]:
# Load clean data and original config
df = pd.read_csv('football_data_clean.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

with open('model_config.json') as f:
    config = json.load(f)

FEATURE_COLS = config['feature_cols']
print(f'Loaded {len(df):,} matches and {len(FEATURE_COLS)} original features')

# Load original models to use as baseline
orig_xgb  = joblib.load('model_xgb_1x2.pkl')
orig_home = joblib.load('model_home_goals.pkl')
orig_away = joblib.load('model_away_goals.pkl')
print('Original models loaded for baseline comparison')

---
## Part A — Better features

We add three new features:
- `rest_days` — days since each team's last match (captures fatigue)
- `goal_momentum` — exponentially-weighted recent goal average (captures hot/cold streaks)
- Real H2H stats precomputed for every team pair (fixes the API placeholder bug)

In [ ]:
# Build long-format team records (same as notebook 02)
home_r = df[['date','home_team','home_score','away_score','result']].copy()
home_r.columns = ['date','team','goals_scored','goals_conceded','result']
home_r['won']  = (home_r['result'] == 'home_win').astype(int)
home_r['drew'] = (home_r['result'] == 'draw').astype(int)

away_r = df[['date','away_team','away_score','home_score','result']].copy()
away_r.columns = ['date','team','goals_scored','goals_conceded','result']
away_r['won']  = (away_r['result'] == 'away_win').astype(int)
away_r['drew'] = (away_r['result'] == 'draw').astype(int)

all_r = pd.concat([home_r, away_r]).sort_values(['team','date']).reset_index(drop=True)
all_r['points'] = all_r['won'] * 3 + all_r['drew']

# --- Rest days (days since last match, capped at 30) ---
all_r['prev_date']  = all_r.groupby('team')['date'].shift(1)
all_r['rest_days']  = (all_r['date'] - all_r['prev_date']).dt.days.clip(1, 30).fillna(21)

# --- Goal momentum (EWM — more recent matches count more) ---
all_r['goal_momentum'] = all_r.groupby('team')['goals_scored'].transform(
    lambda x: x.shift(1).ewm(alpha=0.6, min_periods=1).mean()
)
all_r['goal_momentum'] = all_r['goal_momentum'].fillna(all_r['goal_momentum'].mean())

print('New features computed: rest_days, goal_momentum')
print('\nBrazil last 5 records:')
print(all_r[all_r['team']=='Brazil'].tail(5)[
    ['date','goals_scored','rest_days','goal_momentum']].to_string(index=False))

In [ ]:
# Merge new features back to main dataframe
df['match_idx'] = range(len(df))

# Home team new features
home_new = home_r.copy()
home_new['match_idx'] = df['match_idx'].values
home_new['rest_days']      = all_r[all_r['team'].isin(df['home_team'])]['rest_days'].values[:len(df)]
home_new['goal_momentum']  = all_r[all_r['team'].isin(df['home_team'])]['goal_momentum'].values[:len(df)]

# Cleaner approach: filter and sort per team
new_cols = ['rest_days', 'goal_momentum']

home_feats = all_r[all_r['team'].isin(df['home_team'].unique())].copy()
home_feats = home_feats.sort_values(['team','date']).reset_index(drop=True)

# Join on date + team
df_h = df[['match_idx','date','home_team']].copy()
df_h = df_h.merge(
    all_r[['date','team','rest_days','goal_momentum']].rename(
        columns={'team':'home_team','rest_days':'home_rest_days','goal_momentum':'home_goal_momentum'}
    ).drop_duplicates(subset=['date','home_team'], keep='last'),
    on=['date','home_team'], how='left'
)

df_a = df[['match_idx','date','away_team']].copy()
df_a = df_a.merge(
    all_r[['date','team','rest_days','goal_momentum']].rename(
        columns={'team':'away_team','rest_days':'away_rest_days','goal_momentum':'away_goal_momentum'}
    ).drop_duplicates(subset=['date','away_team'], keep='last'),
    on=['date','away_team'], how='left'
)

df = df.merge(df_h[['match_idx','home_rest_days','home_goal_momentum']], on='match_idx', how='left')
df = df.merge(df_a[['match_idx','away_rest_days','away_goal_momentum']], on='match_idx', how='left')

for c in ['home_rest_days','away_rest_days','home_goal_momentum','away_goal_momentum']:
    df[c] = df[c].fillna(df[c].median())

print('New features merged!')
print(df[['home_team','away_team','home_rest_days','away_rest_days',
          'home_goal_momentum','away_goal_momentum']].head(5).to_string(index=False))

In [ ]:
# Precompute H2H stats for ALL team pairs — fixes the API placeholder bug
print('Computing H2H lookup for all team pairs...')

df['_pair'] = df.apply(lambda r: tuple(sorted([r['home_team'],r['away_team']])), axis=1)
h2h_lookup  = {}

for pair in df['_pair'].unique():
    pair_df  = df[df['_pair'] == pair].sort_values('date')
    t1, t2   = pair
    for home in [t1, t2]:
        away    = t2 if home == t1 else t1
        wins    = 0
        goals   = []
        for _, row in pair_df.iterrows():
            goals.append(row['total_goals'])
            if (row['home_team'] == home and row['result'] == 'home_win') or \
               (row['home_team'] == away and row['result'] == 'away_win'):
                wins += 1
        n   = len(pair_df)
        key = f'{home}|{away}'
        h2h_lookup[key] = {
            'win_rate':  round(wins / n, 3) if n > 0 else 0.40,
            'avg_goals': round(float(np.mean(goals)), 3) if goals else 2.50,
            'num_games': n,
        }

df = df.drop(columns=['_pair'])

with open('h2h_stats.json', 'w') as f:
    json.dump(h2h_lookup, f, indent=2)

print(f'H2H stats saved: {len(h2h_lookup):,} team-pair combinations')
print('\nExample — Brazil vs Argentina:')
print('  As home:', h2h_lookup.get('Brazil|Argentina'))
print('  As away:', h2h_lookup.get('Argentina|Brazil'))

In [ ]:
# Update team_stats.json with goal_momentum for each team
with open('team_stats.json') as f:
    team_stats = json.load(f)

latest_momentum = all_r.groupby('team')['goal_momentum'].last()
latest_rest     = all_r.groupby('team').apply(lambda g: (pd.Timestamp.now() - g['date'].max()).days)

for team in team_stats:
    team_stats[team]['goal_momentum'] = round(float(latest_momentum.get(team, 1.3)), 3)
    team_stats[team]['current_rest_days'] = int(min(latest_rest.get(team, 21), 30))

with open('team_stats.json', 'w') as f:
    json.dump(team_stats, f, indent=2)

print('team_stats.json updated with goal_momentum and current_rest_days')

# Define expanded feature set
NEW_FEATURE_COLS = FEATURE_COLS + [
    'home_rest_days',    'away_rest_days',
    'home_goal_momentum','away_goal_momentum',
]
print(f'\nFeatures: {len(FEATURE_COLS)} original + 4 new = {len(NEW_FEATURE_COLS)} total')

In [ ]:
# Load feature dataset (from notebook 02) and merge new features
df_feat = pd.read_csv('features_dataset.csv')
df_feat['date'] = pd.to_datetime(df_feat['date'])

# Join new features from df (same rows, same order after sort)
new_feat_cols = ['match_idx','home_rest_days','away_rest_days',
                  'home_goal_momentum','away_goal_momentum']
df_feat = df_feat.merge(df[new_feat_cols], on='match_idx', how='left')

for c in ['home_rest_days','away_rest_days','home_goal_momentum','away_goal_momentum']:
    df_feat[c] = df_feat[c].fillna(df_feat[c].median())

df_feat = df_feat.dropna(subset=NEW_FEATURE_COLS + ['result_label']).reset_index(drop=True)

# Train / test split (same as notebook 03)
train_mask = df_feat['date'] < '2022-01-01'
test_mask  = (df_feat['tournament'] == 'FIFA World Cup') & (df_feat['year'] == 2022)
if test_mask.sum() == 0:
    test_mask  = (df_feat['tournament'] == 'FIFA World Cup') & (df_feat['year'] == 2018)
    train_mask = df_feat['date'] < '2018-01-01'

X_train = df_feat.loc[train_mask, NEW_FEATURE_COLS].reset_index(drop=True)
X_test  = df_feat.loc[test_mask,  NEW_FEATURE_COLS].reset_index(drop=True)
y_train = df_feat.loc[train_mask, 'result_label'].reset_index(drop=True)
y_test  = df_feat.loc[test_mask,  'result_label'].reset_index(drop=True)
y_train_hg = df_feat.loc[train_mask,'home_score'].reset_index(drop=True)
y_train_ag = df_feat.loc[train_mask,'away_score'].reset_index(drop=True)
df_test    = df_feat.loc[test_mask].reset_index(drop=True)

print(f'Train: {len(X_train):,}   Test (WC): {len(X_test):,}')

# Baseline: original model on original features
orig_feat_test  = X_test[FEATURE_COLS]
orig_pred_proba = orig_xgb.predict_proba(orig_feat_test)
orig_acc        = accuracy_score(y_test, np.argmax(orig_pred_proba, axis=1))
orig_ll         = log_loss(y_test, orig_pred_proba)
print(f'\nBASELINE (original model): accuracy={orig_acc*100:.1f}%   log_loss={orig_ll:.4f}')

---
## Part B — Hyperparameter tuning with Optuna

Instead of guessing parameters, Optuna runs 75 trials and finds the best combination automatically. Uses time-series cross-validation to avoid looking at the future.

In [ ]:
print('Running Optuna search (75 trials, ~10 minutes)...')

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.4),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 2.0),
    }
    model  = XGBClassifier(**params, random_state=42, n_jobs=-1, verbosity=0)
    tscv   = TimeSeriesSplit(n_splits=3)
    X_arr  = X_train.values
    y_arr  = y_train.values
    scores = []
    for tr_i, val_i in tscv.split(X_arr):
        model.fit(X_arr[tr_i], y_arr[tr_i])
        scores.append(accuracy_score(y_arr[val_i], model.predict(X_arr[val_i])))
    return float(np.mean(scores))

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=75, show_progress_bar=True)

best_params = study.best_params
print(f'\nBest CV accuracy: {study.best_value*100:.2f}%')
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Train tuned XGBoost on full training set
model_tuned = XGBClassifier(**best_params, random_state=42, n_jobs=-1, verbosity=0)
model_tuned.fit(X_train, y_train)

# Also tune Poisson goal models
POISSON_PARAMS = dict(
    objective='count:poisson',
    n_estimators=best_params.get('n_estimators', 300),
    max_depth=best_params.get('max_depth', 4),
    learning_rate=best_params.get('learning_rate', 0.05),
    subsample=best_params.get('subsample', 0.8),
    colsample_bytree=best_params.get('colsample_bytree', 0.8),
    random_state=42, n_jobs=-1, verbosity=0,
)
model_home_tuned = XGBRegressor(**POISSON_PARAMS)
model_away_tuned = XGBRegressor(**POISSON_PARAMS)
model_home_tuned.fit(X_train, y_train_hg)
model_away_tuned.fit(X_train, y_train_ag)

tuned_proba = model_tuned.predict_proba(X_test)
tuned_acc   = accuracy_score(y_test, np.argmax(tuned_proba, axis=1))
tuned_ll    = log_loss(y_test, tuned_proba)
print(f'Tuned XGBoost + new features: accuracy={tuned_acc*100:.1f}%   log_loss={tuned_ll:.4f}')
print(f'Improvement vs baseline:      +{(tuned_acc-orig_acc)*100:.1f}%')

---
## Part C — Dixon-Coles correction

The standard Poisson model underestimates 0-0 and 1-0 results and overestimates 2-1 and 3-1. The Dixon-Coles correction factor (rho) adjusts the score matrix for these low-scoring outcomes, directly improving Over/Under and correct score accuracy.

In [ ]:
def tau(x, y, lh, la, rho):
    """Dixon-Coles adjustment for low-scoring results."""
    if x == 0 and y == 0:   return 1 - lh * la * rho
    elif x == 0 and y == 1: return 1 + lh * rho
    elif x == 1 and y == 0: return 1 + la * rho
    elif x == 1 and y == 1: return 1 - rho
    else:                    return 1.0

def dc_score_matrix(lh, la, rho=-0.13, max_g=9):
    """Poisson score matrix with Dixon-Coles low-score correction."""
    mat = np.outer(
        [poisson.pmf(i, lh) for i in range(max_g)],
        [poisson.pmf(j, la) for j in range(max_g)],
    )
    for i in range(2):
        for j in range(2):
            mat[i, j] *= tau(i, j, lh, la, rho)
    mat /= mat.sum()   # Renormalize so probabilities still sum to 1
    return mat

# Estimate best rho from training data
wc_train = df_feat[train_mask & (df_feat['tournament'] == 'FIFA World Cup')].copy()

def neg_log_likelihood(rho_val, data):
    total = 0
    feat  = data[NEW_FEATURE_COLS].values
    lh_arr = model_home_tuned.predict(feat)
    la_arr = model_away_tuned.predict(feat)
    for i, row in data.reset_index(drop=True).iterrows():
        mat   = dc_score_matrix(lh_arr[i], la_arr[i], rho=rho_val)
        h, a  = int(row['home_score']), int(row['away_score'])
        if h < 9 and a < 9 and mat[h, a] > 0:
            total -= np.log(mat[h, a])
    return total

from scipy.optimize import minimize_scalar
res = minimize_scalar(lambda r: neg_log_likelihood(r, wc_train), bounds=(-0.3, 0.0), method='bounded')
RHO = round(float(res.x), 3)
print(f'Optimal Dixon-Coles rho = {RHO}  (typical range: -0.05 to -0.20)')
print('Rho < 0 confirms the model over-predicts high-scoring results in WC')

# Verify: compare standard vs DC for a low-score match
lh_ex, la_ex = 1.2, 0.9
standard_mat = np.outer([poisson.pmf(i,lh_ex) for i in range(5)],
                         [poisson.pmf(j,la_ex) for j in range(5)])
dc_mat       = dc_score_matrix(lh_ex, la_ex, rho=RHO, max_g=5)
print(f'\n0-0 probability — Standard: {standard_mat[0,0]*100:.1f}%   Dixon-Coles: {dc_mat[0,0]*100:.1f}%')
print(f'1-0 probability — Standard: {standard_mat[1,0]*100:.1f}%   Dixon-Coles: {dc_mat[1,0]*100:.1f}%')

---
## Part D — Probability calibration

XGBoost probabilities can be overconfident — saying 85% when the true probability is 70%. Isotonic regression calibration fixes this without changing accuracy, making the probability numbers themselves more trustworthy.

In [ ]:
# Use 2020-2021 data as calibration set (held out from tuning)
cal_mask = (df_feat['date'] >= '2020-01-01') & (df_feat['date'] < '2022-01-01')
X_cal    = df_feat.loc[cal_mask, NEW_FEATURE_COLS]
y_cal    = df_feat.loc[cal_mask, 'result_label']

print(f'Calibration set: {len(X_cal):,} matches (2020-2021)')

# Calibrate using isotonic regression
model_cal = CalibratedClassifierCV(model_tuned, method='isotonic', cv='prefit')
model_cal.fit(X_cal, y_cal)

# Compare raw vs calibrated probabilities
raw_proba = model_tuned.predict_proba(X_test)
cal_proba = model_cal.predict_proba(X_test)

cal_acc = accuracy_score(y_test, np.argmax(cal_proba, axis=1))
cal_ll  = log_loss(y_test, cal_proba)
raw_ll  = log_loss(y_test, raw_proba)

print(f'\nTuned XGBoost (raw):        accuracy={tuned_acc*100:.1f}%   log_loss={raw_ll:.4f}')
print(f'Tuned XGBoost (calibrated): accuracy={cal_acc*100:.1f}%   log_loss={cal_ll:.4f}')
print(f'Log loss improvement: {raw_ll - cal_ll:.4f}  (lower is better)')

---
## Part E — Model stacking

Instead of manually blending XGBoost and Poisson with fixed weights, a logistic regression meta-model learns the optimal blend. It discovers, for example, that the Poisson model is more reliable for close matchups while XGBoost dominates for big Elo differences.

In [ ]:
print('Building stacked ensemble...')

# Helper: derive 1X2 probabilities from Poisson model
def poisson_1x2(X, home_model, away_model, rho=RHO, max_g=9):
    lh_arr = home_model.predict(X)
    la_arr = away_model.predict(X)
    proba  = np.zeros((len(X), 3))
    for i, (lh, la) in enumerate(zip(lh_arr, la_arr)):
        mat = dc_score_matrix(lh, la, rho=rho, max_g=max_g)
        proba[i, 2] = float(np.sum(np.tril(mat, -1)))   # home win
        proba[i, 1] = float(np.sum(np.diag(mat)))       # draw
        proba[i, 0] = float(np.sum(np.triu(mat, 1)))    # away win
    return proba

# Get out-of-fold (OOF) predictions to avoid leakage in meta-model training
tscv        = TimeSeriesSplit(n_splits=5)
xgb_oof     = np.zeros((len(X_train), 3))
poisson_oof = np.zeros((len(X_train), 3))

for tr_idx, val_idx in tscv.split(X_train):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr        = y_train.iloc[tr_idx]
    y_tr_hg     = y_train_hg.iloc[tr_idx]
    y_tr_ag     = y_train_ag.iloc[tr_idx]

    m_xgb  = XGBClassifier(**best_params, random_state=42, n_jobs=-1, verbosity=0)
    m_home = XGBRegressor(**POISSON_PARAMS)
    m_away = XGBRegressor(**POISSON_PARAMS)
    m_xgb.fit(X_tr, y_tr)
    m_home.fit(X_tr, y_tr_hg)
    m_away.fit(X_tr, y_tr_ag)

    xgb_oof[val_idx]     = m_xgb.predict_proba(X_val)
    poisson_oof[val_idx] = poisson_1x2(X_val, m_home, m_away)

# Train meta-model on OOF predictions
meta_X_train = np.hstack([xgb_oof, poisson_oof])   # 6 features: [away,draw,home] x 2
meta_model   = LogisticRegression(C=1.0, max_iter=500, random_state=42)
meta_model.fit(meta_X_train, y_train)

# Evaluate stacked model on test set
xgb_test     = model_cal.predict_proba(X_test)
poisson_test = poisson_1x2(X_test, model_home_tuned, model_away_tuned)
meta_X_test  = np.hstack([xgb_test, poisson_test])
stacked_proba = meta_model.predict_proba(meta_X_test)

stack_acc = accuracy_score(y_test, np.argmax(stacked_proba, axis=1))
stack_ll  = log_loss(y_test, stacked_proba)
print(f'Stacked ensemble: accuracy={stack_acc*100:.1f}%   log_loss={stack_ll:.4f}')
print(f'Improvement vs tuned+calibrated: +{(stack_acc-cal_acc)*100:.1f}%')

---
## Part F — Neural network (MLP)

A 3-layer MLP adds a different type of model to the ensemble. Neural networks learn feature interactions that tree models sometimes miss. We add its predictions to the meta-learner blend.

In [ ]:
print('Training MLP neural network...')

# Scale features — essential for neural networks
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

model_nn = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    alpha=0.001,               # L2 regularization
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=25,
    random_state=42,
)
model_nn.fit(X_train_sc, y_train)

nn_proba = model_nn.predict_proba(X_test_sc)
nn_acc   = accuracy_score(y_test, np.argmax(nn_proba, axis=1))
nn_ll    = log_loss(y_test, nn_proba)
print(f'Neural network: accuracy={nn_acc*100:.1f}%   log_loss={nn_ll:.4f}')
print(f'Stopped at iteration: {model_nn.n_iter_}')

In [ ]:
# Build final ensemble: XGBoost + Poisson-DC + Neural network
# Get OOF predictions from NN to train final meta-model
nn_oof = np.zeros((len(X_train), 3))
for tr_idx, val_idx in TimeSeriesSplit(n_splits=5).split(X_train_sc):
    m = MLPClassifier(hidden_layer_sizes=(128,64,32), activation='relu',
                       alpha=0.001, max_iter=500, early_stopping=True,
                       validation_fraction=0.1, n_iter_no_change=25, random_state=42)
    m.fit(X_train_sc[tr_idx], y_train.iloc[tr_idx])
    nn_oof[val_idx] = m.predict_proba(X_train_sc[val_idx])

# Final meta-features: XGBoost + Poisson + NN
meta_X_train_final = np.hstack([xgb_oof, poisson_oof, nn_oof])
meta_X_test_final  = np.hstack([xgb_test, poisson_test, nn_proba])

meta_final = LogisticRegression(C=1.0, max_iter=500, random_state=42)
meta_final.fit(meta_X_train_final, y_train)

final_proba = meta_final.predict_proba(meta_X_test_final)
final_acc   = accuracy_score(y_test, np.argmax(final_proba, axis=1))
final_ll    = log_loss(y_test, final_proba)
print(f'FINAL ensemble (XGB+Poisson+NN): accuracy={final_acc*100:.1f}%   log_loss={final_ll:.4f}')

---
## Full before vs after comparison

In [ ]:
results = {
    'Original (NB03)':          (orig_acc,   orig_ll),
    'New features only':        (accuracy_score(y_test, model_tuned.predict(X_test)), log_loss(y_test, raw_proba)),
    '+ Optuna tuning':          (tuned_acc,  tuned_ll),
    '+ Calibration':            (cal_acc,    cal_ll),
    '+ Stacking (XGB+Poisson)': (stack_acc,  stack_ll),
    'Neural network alone':     (nn_acc,     nn_ll),
    'FINAL (XGB+Poisson+NN)':   (final_acc,  final_ll),
}

print(f'{"Model":<30}  {"Accuracy":>9}  {"Log Loss":>10}  {"vs Baseline":>13}')
print('-' * 68)
for name, (acc, ll) in results.items():
    delta = f'+{(acc-orig_acc)*100:.1f}%' if acc > orig_acc else f'{(acc-orig_acc)*100:.1f}%'
    marker = ' <-- BEST' if acc == max(a for a, _ in results.values()) else ''
    print(f'{name:<30}  {acc*100:>8.1f}%  {ll:>10.4f}  {delta:>13}{marker}')

In [ ]:
# Visualize the improvement
names = list(results.keys())
accs  = [v[0]*100 for v in results.values()]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#B5B0E8' if n != 'FINAL (XGB+Poisson+NN)' else '#534AB7' for n in names]
bars = ax.barh(names, accs, color=colors, edgecolor='white')
ax.axvline(orig_acc*100, color='red', linestyle='--', alpha=0.6, linewidth=1.5, label='Baseline')
for bar, val in zip(bars, accs):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Accuracy on 2022 World Cup (%)')
ax.set_title('Model improvement at each stage', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('improvement_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save all improved models

In [ ]:
import shutil

# Save models
joblib.dump(model_tuned,      'model_xgb_tuned.pkl')
joblib.dump(model_cal,        'model_xgb_calibrated.pkl')
joblib.dump(model_home_tuned, 'model_home_goals_v2.pkl')
joblib.dump(model_away_tuned, 'model_away_goals_v2.pkl')
joblib.dump(model_nn,         'model_nn.pkl')
joblib.dump(scaler,           'scaler_nn.pkl')
joblib.dump(meta_final,       'model_meta_final.pkl')

# Save Dixon-Coles rho and new config
config['feature_cols']  = NEW_FEATURE_COLS
config['dc_rho']        = RHO
config['model_version'] = 'v2'
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Copy updated files to project folder
for fname in ['model_xgb_calibrated.pkl','model_home_goals_v2.pkl',
               'model_away_goals_v2.pkl','model_nn.pkl','scaler_nn.pkl',
               'model_meta_final.pkl']:
    shutil.copy2(os.path.join(SAVE_DIR, fname), os.path.join(PROJECT_DIR,'models',fname))

for fname in ['model_config.json','team_stats.json','h2h_stats.json']:
    shutil.copy2(os.path.join(SAVE_DIR, fname), os.path.join(PROJECT_DIR,'data',fname))

print('All models and data files saved and copied to API project folder!')
print()
print('Saved to models/:')
for f in os.listdir(os.path.join(PROJECT_DIR,'models')):
    sz = os.path.getsize(os.path.join(PROJECT_DIR,'models',f))
    print(f'  {f}  ({sz/1024:.0f} KB)')

---
## Update predictor.py with all improvements

In [ ]:
%%writefile /content/drive/MyDrive/StatPitch/statpitch_api/predictor.py
# predictor.py  v2  --  improved: H2H fix, rest days, goal momentum,
#                        Dixon-Coles, calibration, stacking, neural network
import joblib, json
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize_scalar
from pathlib import Path
from datetime import datetime

BASE = Path(__file__).parent

# ─── Load models ─────────────────────────────────────────────────────────────
_xgb   = joblib.load(BASE / 'models' / 'model_xgb_calibrated.pkl')
_home  = joblib.load(BASE / 'models' / 'model_home_goals_v2.pkl')
_away  = joblib.load(BASE / 'models' / 'model_away_goals_v2.pkl')
_nn    = joblib.load(BASE / 'models' / 'model_nn.pkl')
_sc    = joblib.load(BASE / 'models' / 'scaler_nn.pkl')
_meta  = joblib.load(BASE / 'models' / 'model_meta_final.pkl')

# ─── Load data ───────────────────────────────────────────────────────────────
with open(BASE / 'data' / 'model_config.json') as f:
    _cfg = json.load(f)
with open(BASE / 'data' / 'team_stats.json') as f:
    TEAM_STATS = json.load(f)
with open(BASE / 'data' / 'h2h_stats.json') as f:
    H2H_STATS = json.load(f)

FEATURE_COLS = _cfg['feature_cols']
ELO_DEFAULT  = float(_cfg.get('elo_start', 1500))
DC_RHO       = float(_cfg.get('dc_rho', -0.13))

_AVG = {
    'elo': ELO_DEFAULT,
    'gs_avg5': 1.30,  'gc_avg5': 1.30,  'pts_avg5': 1.20,  'win_rate5': 0.35,
    'gs_avg10': 1.30, 'gc_avg10': 1.30, 'pts_avg10': 1.20, 'win_rate10': 0.35,
    'goal_momentum': 1.30, 'current_rest_days': 14,
}


def get_teams():
    return sorted(TEAM_STATS.keys())


def _rest_days(team, match_date=None):
    if match_date is None:
        match_date = datetime.now()
    elif isinstance(match_date, str):
        match_date = datetime.strptime(match_date, '%Y-%m-%d')
    stats      = TEAM_STATS.get(team, {})
    last_match = stats.get('last_match')
    if last_match is None:
        return 14
    last_dt = datetime.strptime(last_match, '%Y-%m-%d')
    return int(min(max((match_date - last_dt).days, 1), 30))


def _build_features(home, away, is_neutral, match_date=None):
    h  = TEAM_STATS.get(home, _AVG)
    a  = TEAM_STATS.get(away, _AVG)
    he = h.get('elo', ELO_DEFAULT)
    ae = a.get('elo', ELO_DEFAULT)

    h2h_key = f'{home}|{away}'
    h2h     = H2H_STATS.get(h2h_key, {'win_rate': 0.40, 'avg_goals': 2.50, 'num_games': 0})

    h_rest = _rest_days(home, match_date)
    a_rest = _rest_days(away, match_date)

    row = {
        'home_elo':         he,
        'away_elo':         ae,
        'elo_diff':         he - ae,
        'elo_prob_home':    1 / (1 + 10 ** ((ae - he) / 400)),
        'home_gs_avg5':     h.get('gs_avg5',    _AVG['gs_avg5']),
        'home_gc_avg5':     h.get('gc_avg5',    _AVG['gc_avg5']),
        'home_pts_avg5':    h.get('pts_avg5',   _AVG['pts_avg5']),
        'home_win_rate5':   h.get('win_rate5',  _AVG['win_rate5']),
        'home_gs_avg10':    h.get('gs_avg10',   _AVG['gs_avg10']),
        'home_gc_avg10':    h.get('gc_avg10',   _AVG['gc_avg10']),
        'home_pts_avg10':   h.get('pts_avg10',  _AVG['pts_avg10']),
        'home_win_rate10':  h.get('win_rate10', _AVG['win_rate10']),
        'away_gs_avg5':     a.get('gs_avg5',    _AVG['gs_avg5']),
        'away_gc_avg5':     a.get('gc_avg5',    _AVG['gc_avg5']),
        'away_pts_avg5':    a.get('pts_avg5',   _AVG['pts_avg5']),
        'away_win_rate5':   a.get('win_rate5',  _AVG['win_rate5']),
        'away_gs_avg10':    a.get('gs_avg10',   _AVG['gs_avg10']),
        'away_gc_avg10':    a.get('gc_avg10',   _AVG['gc_avg10']),
        'away_pts_avg10':   a.get('pts_avg10',  _AVG['pts_avg10']),
        'away_win_rate10':  a.get('win_rate10', _AVG['win_rate10']),
        'h2h_home_win_rate':h2h['win_rate'],
        'h2h_avg_goals':    h2h['avg_goals'],
        'h2h_num_games':    h2h['num_games'],
        'is_neutral':       int(is_neutral),
        'tournament_weight':1.00,
        'home_rest_days':   h_rest,
        'away_rest_days':   a_rest,
        'home_goal_momentum': h.get('goal_momentum', _AVG['goal_momentum']),
        'away_goal_momentum': a.get('goal_momentum', _AVG['goal_momentum']),
    }
    return np.array([[row[c] for c in FEATURE_COLS]])


def _tau(x, y, lh, la, rho):
    if x == 0 and y == 0:   return 1 - lh * la * rho
    elif x == 0 and y == 1: return 1 + lh * rho
    elif x == 1 and y == 0: return 1 + la * rho
    elif x == 1 and y == 1: return 1 - rho
    else:                    return 1.0


def _dc_matrix(lh, la, max_g=9):
    mat = np.outer(
        [poisson.pmf(i, lh) for i in range(max_g)],
        [poisson.pmf(j, la) for j in range(max_g)],
    )
    for i in range(2):
        for j in range(2):
            mat[i, j] *= _tau(i, j, lh, la, DC_RHO)
    mat /= mat.sum()
    return mat


def _markets(lh, la, max_g=9):
    mat  = _dc_matrix(lh, la, max_g)
    idx  = np.array([[i + j for j in range(max_g)] for i in range(max_g)])
    btts = float((1 - poisson.pmf(0, lh)) * (1 - poisson.pmf(0, la)))
    top  = sorted(
        [{'score': f'{i}-{j}', 'probability': round(float(mat[i, j]), 4)}
         for i in range(max_g) for j in range(max_g)],
        key=lambda x: x['probability'], reverse=True,
    )[:10]
    return {
        'match_result': {
            'home_win': round(float(np.sum(np.tril(mat, -1))), 4),
            'draw':     round(float(np.sum(np.diag(mat))),     4),
            'away_win': round(float(np.sum(np.triu(mat, 1))),  4),
        },
        'over_under': {
            'over_1_5': round(float(np.sum(mat[idx > 1])), 4),
            'over_2_5': round(float(np.sum(mat[idx > 2])), 4),
            'over_3_5': round(float(np.sum(mat[idx > 3])), 4),
        },
        'btts': {'yes': round(btts, 4), 'no': round(1 - btts, 4)},
        'correct_score': top,
    }


def predict(home, away, is_neutral=True, match_date=None):
    feat   = _build_features(home, away, is_neutral, match_date)
    lh     = float(_home.predict(feat)[0])
    la     = float(_away.predict(feat)[0])

    xgb_p  = _xgb.predict_proba(feat)[0]              # calibrated XGBoost
    poi_mat = _dc_matrix(lh, la)
    poi_p   = np.array([
        float(np.sum(np.triu(poi_mat, 1))),
        float(np.sum(np.diag(poi_mat))),
        float(np.sum(np.tril(poi_mat, -1))),
    ])
    nn_p   = _nn.predict_proba(_sc.transform(feat))[0]

    meta_x = np.hstack([xgb_p, poi_p, nn_p]).reshape(1, -1)
    final_p = _meta.predict_proba(meta_x)[0]           # [away, draw, home]

    out = _markets(lh, la)
    out['match_result'] = {
        'home_win': round(float(final_p[2]), 4),
        'draw':     round(float(final_p[1]), 4),
        'away_win': round(float(final_p[0]), 4),
    }

    h = TEAM_STATS.get(home, _AVG)
    a = TEAM_STATS.get(away, _AVG)
    h2h_key = f'{home}|{away}'
    h2h = H2H_STATS.get(h2h_key, {'win_rate': 0.40, 'avg_goals': 2.50, 'num_games': 0})

    return {
        'home_team': home,
        'away_team': away,
        'expected_goals': {'home': round(lh, 3), 'away': round(la, 3)},
        'team_info': {
            'home_elo':      round(h.get('elo', ELO_DEFAULT), 1),
            'away_elo':      round(a.get('elo', ELO_DEFAULT), 1),
            'elo_diff':      round(h.get('elo', ELO_DEFAULT) - a.get('elo', ELO_DEFAULT), 1),
            'h2h_games':     h2h['num_games'],
            'h2h_home_wins': round(h2h['win_rate'] * 100, 1),
        },
        'model_version': 'v2',
        **out,
    }

---
## Summary

**What was improved and why it matters:**

| Improvement | Technical change | User impact |
|---|---|---|
| H2H fix | Real pair stats from `h2h_stats.json` | API predictions now use actual rivalry history |
| Rest days | Days since last match per team | Fatigue in knockout stages now captured |
| Goal momentum | EWM-weighted recent goals | Hot/cold streaks affect predictions |
| Optuna tuning | 75-trial hyperparameter search | XGBoost now at its optimal configuration |
| Dixon-Coles | Tau correction on score matrix | 0-0, 1-0, 0-1, 1-1 now correctly weighted |
| Calibration | Isotonic regression on held-out set | Probability numbers more trustworthy |
| Stacking | LogReg meta-learner on OOF predictions | Model learns best blend per situation |
| Neural network | 128-64-32 MLP with early stopping | Adds non-tree patterns to ensemble |

**To deploy the improved API:** push the updated `statpitch_api` folder to GitHub — Render will redeploy automatically.

**To keep the model fresh:** re-run this notebook whenever a new qualifying window or tournament finishes. The Optuna and OOF sections will pick up the new data automatically.